## Introduction
This notebook will perform extensive data analysis regarding bird bioacoustic monitoring. The data will be taken from the article "Automated identification of individual birds by song enables multi-year recapture from passive acoustic monitoring data", and it is located under resources/localized_events.csv. 

In [ ]:
# First, we must import the necessary packages and libraries
import pandas as pd
import ast
import matplotlib as mpl
import numpy as np
from pathlib import Path 
from plotnine import ( 
    ggplot, 
    aes, 
    geom_point,
    geom_bar,
    facet_wrap,
    labs,
    scale_fill_manual,
    theme_minimal
)

Then, we will need to load in the data as a dataframe or just fetch the binary if it exists

In [ ]:
path = Path("resources/localized_events.parquet")

if path.exists():
    print("Exists, so just load in binary file")
    all_loaded_events_df = pd.read_parquet("resources/localized_events.parquet")
else:
    print("Does not exist, so we create it")
    all_loaded_events_df = pd.read_csv("resources/localized_events.csv")
    all_loaded_events_df.to_parquet("resources/localized_events.parquet")


Now, we will take a subset of the column variables in the data spreadsheet.

In [ ]:
subst_loaded_events_df = all_loaded_events_df[
    [
    # Metadata
    'array',
    'data_split',
    'relative_position',
    'mic_distances_m',
    'start_timestamp',
    'duration',

    # Automated models
    'ovenbird_cluster_labels',
    'baseline_resnet18_cluster_labels',
    'hawkears_cluster_labels',
    'birdnet_cluster_labels',
    ]
]

### Exploratory Data Analysis
We will begin by exploring the distributions of the data to identify possible outliers, trends, and occurrences. 

In [ ]:
# add a new column individual so it does not conflict with python
subst_loaded_events_df['individual'] = all_loaded_events_df['class']

# Convert to datetime objects
subst_loaded_events_df['start_timestamp'] = pd.to_datetime(subst_loaded_events_df['start_timestamp'], utc=True)
subst_loaded_events_df['duration'] = pd.to_timedelta(subst_loaded_events_df['duration'], unit='minute')

In [ ]:
# Create a new column just for the end_timestamp
subst_loaded_events_df['end_timestamp'] = (subst_loaded_events_df['start_timestamp'] + subst_loaded_events_df['duration'])

In [ ]:
# Set an event key for all the rows
subst_loaded_events_df['event_key'] = range(len(subst_loaded_events_df))

In [ ]:
# .value_counts() creates a lookup table, (individual: counts) for all individuals
# .map() takes that lookup table and assigns it to those individuals using 'individual' as the key
# This new column 'count' is assigned with that counts for each individual
lookup_individual_counts = subst_loaded_events_df['individual'].value_counts()
subst_loaded_events_df['count'] = subst_loaded_events_df['individual'].map(
    lookup_individual_counts
)

In [ ]:
# Save to under resources
subst_loaded_events_df.to_parquet('resources/subset_loaded_events.parquet')
subst_loaded_events_df.to_csv('resources/subset_loaded_events.csv')

In [ ]:
# Creates a ggplot object 
(
    ggplot(subst_loaded_events_df, aes(x='factor(individual)', fill='count > 30'))
    + geom_bar()
    + scale_fill_manual(
        values={True: 'blue', False: 'red'}
    )
    + labs(
        title='Song Count per Individual',
        x='Individual',
        y='Count'
    )
)

Now, we want to plot the recordings according to the relative (x, y) coordinates, they will be labeled according to the individual they belong to & faceted by the sites they belong to.

In [ ]:
# Create the figure for this, using ast package to unpack the str
fig_df = pd.DataFrame()
fig_df['x'] = subst_loaded_events_df['relative_position'].apply(
    lambda p: ast.literal_eval(p)[0]
)
fig_df['y'] = subst_loaded_events_df['relative_position'].apply(
    lambda p: ast.literal_eval(p)[1]
)
fig_df['individual'] = subst_loaded_events_df['individual'].astype("category")

In [ ]:
(
    ggplot(fig_df, aes('x', 'y', fill = 'individual')) +
        geom_point() 
)

Now, we will facet grid them by the site.

In [ ]:
# Introduce the site column to enable facet gridding
fig_df['site'] = subst_loaded_events_df['array']

In [ ]:
(
    ggplot(fig_df, aes('x', 'y', fill = 'individual')) +
        geom_point() +
        facet_wrap('site') +
        labs(
        title='Ovenbird Song Positions by Array',
        x='Easting (m)',
        y='Northing (m)',
        color='Individual'    
        ) +
        theme_minimal()
)

### Bird Recordings Overlap
The overlaps of all the bird recordings will be found, and that information will be used to determine bird interactions.
- If two bird recordings overlap, that may or may not be an interaction (further analysis is required)
- If two bird recordings do not overlap, that is not an interaction

In [ ]:
#subst_loaded_events_df.iloc[0]
len(subst_loaded_events_df)

In [ ]:
from pandas import DataFrame

def create_tuple_from_df(df: DataFrame, start: str, end: str):
    return list(df[[start, end]].itertuples(index=True, name=None))

In [ ]:
# Create tuple events from the dataframe
tuples_events = create_tuple_from_df(subst_loaded_events_df, 'start_timestamp', 'end_timestamp')
tuples_events

In [ ]:
# Find the overlap from all the events
def overlap(events) -> list[tuple]:
    # We need to find all the ways the events overlap, then return a tuple of event ids 
    events_overlapped: list[tuple] = []
    events = sorted(events, key=lambda x: x[1]) # key must use a function for comparison

    for i, (key_1, _, end_1) in enumerate(events): # We do not care about the start for current event due to sort
        for key_2, start_2, _ in events[i+1:]:  # compare against all later events

            if start_2 < end_1:  # overlap condition
                events_overlapped.append((key_1, key_2))
            else: # start_2 >= end_1, which means there is no overlap
                break  # Sorting invariant, can break out early for speedups
            
    return events_overlapped

In [ ]:
# Test that sequence of events
overlaps = overlap(tuples_events)
overlaps

Naively it works, but the overlap count appears to be very low. This suggests that the individual birds in the sites typically do not interrupt one another.

Now, we should specify upper and lower bounds to increase the overlaps. Eg. Two events overlap if they occurred within x minutes of one another.

In [ ]:
# First we need to add a lower and upper bound: Start - Lower & End + Upper. Then, use that for overlaps
# Modify the dataframe, adding those new columns for the algorithm to process 
cols = subst_loaded_events_df.columns
#cols
# Fetch only the necessary columns 
time_df = subst_loaded_events_df[['event_key', 'start_timestamp', 'end_timestamp']]
# Process those columns by adding a new one

lower_s = 10
upper_e = 10

time_df['scale_time_start'] = time_df['start_timestamp'] - pd.to_timedelta(lower_s, unit='minute')
time_df['scale_time_end'] = time_df['end_timestamp'] - pd.to_timedelta(upper_e, unit='minute')

time_df = time_df.set_index("event_key")


At this point, we have sucessfully transformed those columns by adding and subtracting increments

In [ ]:
time_df

In [ ]:
tuples_events_scale = create_tuple_from_df(time_df, 'start_timestamp', 'end_timestamp')

Now that we have successfully transformed the original dataframe to reflect a broader range, we will run the overlap algorithm on this.

In [ ]:
event_overlapping_ids = overlap(tuples_events_scale) # We now have a list of overlapping event ids
# {event_key, event_1, event_2}
event_overlapping_ids

In [ ]:
list(subst_loaded_events_df.columns)

### Identifying Individuals
Using the unique event ids, we will find the corresponding individuals and plot only those events that occurred at the same time.

In [ ]:
# Fetch the rows that correspond to the individuals that have overlapped
individuals = [] # (individual_1, individual_2)
for id in event_overlapping_ids:
    e_1 = subst_loaded_events_df.loc[id[0]]
    e_2 = subst_loaded_events_df.loc[id[1]]

    ind_event_1 = e_1["individual"]
    ind_event_2 = e_2["individual"]

    b_1 = int(ind_event_1)  # type: ignore
    b_2 = int(ind_event_2) # type: ignore

    if e_1["array"] != e_2["array"] or b_1 == b_2: # Check that the site is the same and bird is unique
        continue 
    
    individuals.append( # append the bird & it's row info so we do not have to search again
        (
        {
            "name": b_1,
            "info": e_1
        },
        {
            "name": b_2,
            "info": e_2
        }
        )
    )

In [ ]:
individuals

Now we have a list of the individuals with overlapping acoustic records, we have enough information to create visualization of the graphs.

Import the networkX library in order to store the individuals into the graph.

In [ ]:
import networkx as nx

In [ ]:
# Now we can iterate over the list of tuples, adding them to the Graph
# ({name, info}, {name, info}) : These two individual birds had an event overlap, each with its own metadata
bird_net = nx.Graph()
#individuals

for overlap in individuals:
    a = overlap[0]["name"]
    b = overlap[1]["name"]
    
    bird_net.add_node(a)
    bird_net.add_node(b)

    if bird_net.has_edge(a, b):
        bird_net[a][b]["weight"] += 1  # increment on repeat overlap
    else:
        bird_net.add_edge(a, b, weight=1)

In [ ]:
print(len(individuals))
print(bird_net.number_of_edges())

In [ ]:
for u, v, data in bird_net.edges(data=True):
    print(u, v, data)
    break

In [ ]:
print(max(data["weight"] for _, _, data in bird_net.edges(data=True)))

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

components = list(nx.connected_components(bird_net))

cmap = plt.get_cmap("tab20") # 10 distinct colors

# Map each node to its component's color
color_map = {}
for i, component in enumerate(components):
    for node in component:
        color_map[node] = cmap(i % 10)

node_colors = [color_map[node] for node in bird_net.nodes()]
pos = nx.kamada_kawai_layout(bird_net, scale=5, weight=None)      

weights = [bird_net[u][v]["weight"] for u, v in bird_net.edges()]
max_weight = max(weights)
norm_weights = [w / max_weight * 5 for w in weights]  # scale to max thickness of 5

plt.figure(figsize=(25, 20))  # also make the figure itself bigger
nx.draw(bird_net, pos,
        with_labels=True,
        node_color=node_colors,
        edge_color="gray",
        node_size=700,        # bigger nodes
        font_size=8,
        font_color="white",
        width=norm_weights)

plt.show()

In [ ]:
components = list(nx.connected_components(bird_net))
for i, comp in enumerate(components):
    print(f"Component {i}: {sorted(comp)}")

In [ ]:
pos = {}
components = list(nx.connected_components(bird_net))

# Place each component in its own region
cols = 4  # number of columns in the grid layout

for i, component in enumerate(components):
    subgraph = bird_net.subgraph(component)
    if len(component) == 2:
        nodes = list(component)
        sub_pos = {nodes[0]: np.array([-1.5, 0]), nodes[1]: np.array([1.5, 0])}
    else: 
        scale = max(2, len(component) * 0.5)  # bigger components get more space
        sub_pos = nx.kamada_kawai_layout(subgraph, scale=scale)
    
    # Offset each component into a grid position
    row, col = divmod(i, cols)
    offset = np.array([col * 8, -row * 8])
    for node, p in sub_pos.items():
        pos[node] = p + offset

plt.figure(figsize=(15, 10))
nx.draw(bird_net, pos,
        with_labels=True,
        node_color=node_colors,
        edge_color="gray",
        node_size=300,
        font_size=8,
        font_color="white")
plt.show()

In [ ]:
from pyvis.network import Network
import networkx as nx

In [ ]:
# Build a site -> color mapping
site_map = {}  # node -> site
for node in bird_net.nodes():
    site_map[node] = subst_loaded_events_df[
        subst_loaded_events_df["individual"] == node
    ].iloc[0]["array"]

sites = list(set(site_map.values()))
site_colors = {site: cmap(i % 10) for i, site in enumerate(sites)}

In [ ]:
net = Network(notebook=True, height="750px", width="100%")

# Add nodes
for i, component in enumerate(components):
    site = site_map[node]
    color = site_colors[site]
    hex_color = "#{:02x}{:02x}{:02x}".format(
        int(color[0]*255), int(color[1]*255), int(color[2]*255)
    )
    for node in component:
        net.add_node(node,
                     label=str(node),
                     color=hex_color,
                     font={"size": 14, "color": "black", "align": "center"})

# Add edges with normalized weights
weights = [data.get("weight", 1) for _, _, data in bird_net.edges(data=True)]
min_w, max_w = min(weights), max(weights)

for src, dest, data in bird_net.edges(data=True):
    w = data.get("weight", 1)
    normalized = 1 + 14 * (w - min_w) / (max_w - min_w)
    net.add_edge(src, dest, value=normalized)

net.show_buttons(filter_=["physics"])
net.show("bird_net.html")

In [ ]:
print(len(sites))
print(sites)

In [236]:
all_sites = subst_loaded_events_df["array"].unique()
graph_sites = set(site_map.values())
print(set(all_sites) - graph_sites)  # sites missing from graph

{'SBT-3-15', 'SBT-6-83'}
